In [ ]:
import numpy as np
import pandas as pd


In [ ]:
from tensorflow.keras import layers , models
from sklearn.utils.class_weight import compute_class_weight
import cv2 
from tensorflow.keras.applications import ResNet152V2

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir="skin-disease-datasaet/train_set"
test_dir="skin-disease-datasaet/test_set"

img_size = 224
batch_size = 32


data_gen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.6, 1.4],
    fill_mode='nearest',
    validation_split=0.25
)



train_gen = data_gen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    interpolation='bilinear',
    shuffle=True
)

valid_gen = data_gen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    interpolation='bilinear',
    shuffle=False
)

print("Class indices:", train_gen.class_indices)
print("Class sample counts:", np.bincount(train_gen.classes))

Found 695 images belonging to 8 classes.
Found 229 images belonging to 8 classes.
Class indices: {'BA- cellulitis': 0, 'BA-impetigo': 1, 'FU-athlete-foot': 2, 'FU-nail-fungus': 3, 'FU-ringworm': 4, 'PA-cutaneous-larva-migrans': 5, 'VI-chickenpox': 6, 'VI-shingles': 7}
Class sample counts: [102  60  93  97  68  75 102  98]


Found 924 images belonging to 8 classes.
Found 183 images belonging to 8 classes.
Found 233 images belonging to 8 classes.


In [16]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import ResNet50

In [27]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weights_dict)

Class Weights: {0: 0.8517156862745098, 1: 1.4479166666666667, 2: 0.9341397849462365, 3: 0.895618556701031, 4: 1.2775735294117647, 5: 1.1583333333333334, 6: 0.8517156862745098, 7: 0.8864795918367347}


In [26]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

base_model = ResNet152V2(include_top=False, input_shape=(img_size, img_size, 3))
base_model.trainable = False  

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

234545216/234545216 [==============================] - 98s 0us/step
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet152v2 (Functional)    (None, 7, 7, 2048)        58331648  
                                                                 
 global_average_pooling2d_4  (None, 2048)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dense_15 (Dense)            (None, 256)               524544    
                                                                 
 dropout_6 (Dropout)         (None, 256)               0         
                                                                 
 dense_16 (Dense)            (None, 8)                 2056      
                                                                 
Total params: 58858248 (224.53 MB)
Trainable params:

In [28]:
history=model.fit(
    train_gen,
    epochs=25,
    validation_data=valid_gen,
    class_weight=class_weights_dict
)

Epoch 1/25
22/22 [==============================] - 25s 839ms/step - loss: 1.7769 - accuracy: 0.5122 - val_loss: 0.8811 - val_accuracy: 0.7642
Epoch 2/25
22/22 [==============================] - 13s 607ms/step - loss: 0.6411 - accuracy: 0.7813 - val_loss: 0.6883 - val_accuracy: 0.7773
Epoch 3/25
22/22 [==============================] - 13s 607ms/step - loss: 0.5751 - accuracy: 0.8187 - val_loss: 0.5992 - val_accuracy: 0.8166
Epoch 4/25
22/22 [==============================] - 13s 608ms/step - loss: 0.4286 - accuracy: 0.8460 - val_loss: 0.4695 - val_accuracy: 0.8384
Epoch 5/25
22/22 [==============================] - 15s 666ms/step - loss: 0.2891 - accuracy: 0.9137 - val_loss: 0.5068 - val_accuracy: 0.8559
Epoch 6/25
22/22 [==============================] - 21s 944ms/step - loss: 0.3048 - accuracy: 0.8921 - val_loss: 0.4704 - val_accuracy: 0.8472
Epoch 7/25
22/22 [==============================] - 31s 1s/step - loss: 0.3081 - accuracy: 0.8950 - val_loss: 0.3539 - val_accuracy: 0.8777
Ep

In [29]:
model.save("ass3.h5")

/Users/ashishshirsath/tf-cnn-env/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
